### Judging ID-classification accuracy and OOD-generation quality.

tl;dr Quality for both, classification of whether a negative trace contains a calculation error and quality of the error-augmented traces, are quite low. From this I'm drawing three possible patches to apply and retry:

- Use stronger base model for classification/augmentation. I'll make another run, this time using Qwen3-32B.
- Prompt may be too complicated. The prompt may be too complicated. I'll reduce the complexity of the examples and re-generate the base datasets.
- Negative data includes a lot of python-hallucination answers - they seem quite frequent. I wouldn't want to use those anyways, perhaps I can pre-filter those out by removing any trace which contains "```python"
- The data seems to be quite hard, answers are quite long. It may just be that the complexity of this task is too high. I wouldn't want to switch to gsm8k now, lets retry with the above patches and see whether to escalate later.

In [1]:
import numpy as np
import pandas as pd
import os

In [37]:
with open('/u/rfechner/data/ariadne/ood-outputs.parquet', 'rb') as file:
    ooddf = pd.read_parquet(file)

with open('/u/rfechner/data/ariadne/id-outputs.parquet', 'rb') as file:
    iddf = pd.read_parquet(file)

In [38]:
# get random samples for each dataframe

ood_samples = ooddf.sample(n=20, random_state=0)
id_samples = iddf.sample(n=20, random_state=0)

### OOD Analysis

Number of samples: 20
Number of degenerate samples: 20

In [35]:
ood_samples.iloc[19]['responses'][0]

"You are a helpful case generator and an expert in mathematical reasoning. Your task is to decide whether it is reasonable to inject a single, subtle *calculation error* into the provided ground-truth solution and, if so, produce the augmented (erroneous) solution exactly as specified below. If it is NOT reasonable or if there is any ambiguity, you MUST finally output: '#### Not applicable'.auction\nauction\nYou're given a question and a correct student answer, you task is to inject a calculation error if and only if making an error is reasonable in the given context.A calculation error is defined as: 'A calculation error is a mistake made while performing arithmetic or algebraic operations (e.g., adding, subtracting, multiplying, or simplifying incorrectly).'. I expect high precision, please only return an augmented answer if you're really sure to meet conditions. You may always return '#### Not applicable' in case it's impossible to inject an error.Please put a '####' before your fin

### ID-Analysis
Measure the accuracy  of the classification.
Number of samples: 20
Number of correct classifications: 1 (debatable whether this is a spurious success)
Number of degenerate/misunderstood answers: 19

In [66]:
i = 19
print(id_samples.iloc[i]['prompt'][1]['content'])
print("\n\n######################\n\n", id_samples.iloc[i]['responses'][0])

You're given a question and an incorrect students answer, you task is binary classification: Answer whether or not the students answer contains a calculation error. A calculation error is defined as: 'A calculation error is a mistake made while performing arithmetic or algebraic operations (e.g., adding, subtracting, multiplying, or simplifying incorrectly).'. I expect high precision, please only answer 'yes' if you're really sure. In case of ambiguity please return 'no'.Please put your final answer into a \boxed{} latex environment, such that i may parse your answer easily.A positive example:
[QUESTION]
Please calculate the sum of the uneven numbers from 5 to 19 including borders.
[STUDENT]
Okay, to calculate the sum of uneven numbers, i'll add these numbers: 5+7+9+11+13+15+17+19=98. Answer: \boxed{98}
[GT]
96
[ANSWER]
I'm really sure the student made a calculation error, as the solution includes adding up numbers and the final answer is incorrect. The student made a calculation error